<a href="https://colab.research.google.com/github/aarti-raut/AML-Lab-Experiments/blob/main/Experiment_05_Find_S_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 5: Find-S Algorithm
**Aim:** Implement the Find-S algorithm to find the most specific hypothesis.
**Note:** Find-S works on categorical data. We use a simple weather dataset (concept learning).

In [1]:
import pandas as pd
import numpy as np

# Creating a simple concept learning dataset (like PlayTennis / EnjoySport)
# This is the classic dataset used to demonstrate Find-S
data = {
    'Sky':       ['Sunny', 'Sunny', 'Rainy', 'Sunny', 'Sunny', 'Rainy'],
    'AirTemp':   ['Warm',  'Warm',  'Cold',  'Warm',  'Warm',  'Warm'],
    'Humidity':  ['Normal','High',  'High',  'High',  'Normal','Normal'],
    'Wind':      ['Strong','Strong','Strong','Strong','Weak',  'Strong'],
    'Water':     ['Warm',  'Warm',  'Warm',  'Cool',  'Warm',  'Warm'],
    'Forecast':  ['Same',  'Same',  'Change','Change','Same',  'Same'],
    'EnjoySport':['Yes',   'Yes',   'No',    'Yes',   'Yes',   'No']
}

df = pd.DataFrame(data)
print('=== Training Dataset ===')
print(df)


=== Training Dataset ===
     Sky AirTemp Humidity    Wind Water Forecast EnjoySport
0  Sunny    Warm   Normal  Strong  Warm     Same        Yes
1  Sunny    Warm     High  Strong  Warm     Same        Yes
2  Rainy    Cold     High  Strong  Warm   Change         No
3  Sunny    Warm     High  Strong  Cool   Change        Yes
4  Sunny    Warm   Normal    Weak  Warm     Same        Yes
5  Rainy    Warm   Normal  Strong  Warm     Same         No


In [2]:
# Find-S Algorithm Implementation
def find_s_algorithm(data, target_col):
    # Features (all columns except target)
    features = [col for col in data.columns if col != target_col]

    # Start with the most specific hypothesis (null/empty)
    hypothesis = ['0'] * len(features)

    print('\n=== Find-S Algorithm Steps ===')
    print(f'Initial Hypothesis: {hypothesis}\n')

    for idx, row in data.iterrows():
        if row[target_col] == 'Yes':  # Only process positive examples
            print(f'Processing Positive Example {idx + 1}: {list(row[features])}')
            for i, feature in enumerate(features):
                if hypothesis[i] == '0':
                    # Hypothesis was null, adopt this value
                    hypothesis[i] = row[feature]
                elif hypothesis[i] != row[feature]:
                    # Values differ, generalize with '?'
                    hypothesis[i] = '?'
            print(f'Updated Hypothesis:  {hypothesis}\n')

    return features, hypothesis

features, final_hypothesis = find_s_algorithm(df, 'EnjoySport')

print('=== Final Most Specific Hypothesis ===')
for f, h in zip(features, final_hypothesis):
    print(f'  {f}: {h}')



=== Find-S Algorithm Steps ===
Initial Hypothesis: ['0', '0', '0', '0', '0', '0']

Processing Positive Example 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
Updated Hypothesis:  ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']

Processing Positive Example 2: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same']
Updated Hypothesis:  ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']

Processing Positive Example 4: ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change']
Updated Hypothesis:  ['Sunny', 'Warm', '?', 'Strong', '?', '?']

Processing Positive Example 5: ['Sunny', 'Warm', 'Normal', 'Weak', 'Warm', 'Same']
Updated Hypothesis:  ['Sunny', 'Warm', '?', '?', '?', '?']

=== Final Most Specific Hypothesis ===
  Sky: Sunny
  AirTemp: Warm
  Humidity: ?
  Wind: ?
  Water: ?
  Forecast: ?


In [3]:
# Test the hypothesis on new instances
print('\n=== Testing Hypothesis on New Examples ===')

def classify(instance, features, hypothesis):
    for i, feature in enumerate(features):
        if hypothesis[i] == '?':
            continue  # Any value is acceptable
        elif hypothesis[i] == '0':
            return 'No'  # Hypothesis never updated for this feature
        elif instance[feature] != hypothesis[i]:
            return 'No'
    return 'Yes'

test_cases = [
    {'Sky': 'Sunny', 'AirTemp': 'Warm', 'Humidity': 'Normal', 'Wind': 'Strong', 'Water': 'Warm', 'Forecast': 'Same'},
    {'Sky': 'Rainy', 'AirTemp': 'Cold', 'Humidity': 'High',   'Wind': 'Weak',   'Water': 'Cool', 'Forecast': 'Change'},
]

for i, test in enumerate(test_cases):
    result = classify(test, features, final_hypothesis)
    print(f'Test {i+1}: {list(test.values())} => EnjoySport: {result}')



=== Testing Hypothesis on New Examples ===
Test 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'] => EnjoySport: Yes
Test 2: ['Rainy', 'Cold', 'High', 'Weak', 'Cool', 'Change'] => EnjoySport: No
